In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")

# AutoMSARIMAX Model

> Automatic multiple-seasonal SARIMAX model.

`AutoMSARIMAX` searches over bounded AR, differencing, and MA orders at each lag level, then selects the specification with the best information criterion. It fits one SARIMAX model for every candidate order combination, so broad bounds can be slow. Start with restricted bounds and widen them only when needed.


## Simulated Data

The example below simulates hourly data with daily and weekly seasonalities. The final 48 observations are held out to compare the broader automatic search with a cheaper restricted search.

The timing cells call `fit` and `predict` directly so the fitted model metadata, such as `model_` and `selected_orders_`, is available for inspection.


In [ ]:
import time

import numpy as np
import pandas as pd

from statsforecast import StatsForecast
from statsforecast.models import AutoMSARIMAX


def simulate_two_seasonal(n=240, seed=42):
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    daily = 3.0 * np.sin(2 * np.pi * t / 24)
    weekly = 1.8 * np.cos(2 * np.pi * t / 168)
    trend = 0.01 * t
    noise = rng.normal(scale=0.35, size=n)
    return 20 + trend + daily + weekly + noise


h = 48
y = simulate_two_seasonal()
df = pd.DataFrame(
    {
        "unique_id": "series_1",
        "ds": pd.date_range("2024-01-01", periods=len(y), freq="h"),
        "y": y,
    }
)
train_df = df.iloc[:-h]
train_y = train_df["y"].to_numpy()
test = df["y"].to_numpy()[-h:]


## Fuller Automatic Search

This configuration keeps the candidate space modest, but it still fits every valid combination inside the specified bounds. With `max_ar_order=[1, 1, 1]`, `max_i_order=[1, 1, 0]`, `max_ma_order=[0, 0, 0]`, and `include_constant=False`, the search evaluates 31 candidate SARIMAX models.

The number of candidates grows multiplicatively, so adding MA terms or constants can make the same example much slower.


In [ ]:
full_model = AutoMSARIMAX(
    lags=[1, 24, 168],
    max_ar_order=[1, 1, 1],
    max_i_order=[1, 1, 0],
    max_ma_order=[0, 0, 0],
    include_constant=False,
    n_jobs=-1,
)

start = time.perf_counter()
full_model.fit(train_y)
full_fcst = full_model.predict(h=h, level=[80, 95])
full_elapsed = time.perf_counter() - start
full_rmse = np.sqrt(np.mean((full_fcst["mean"] - test) ** 2))

print("selection:", full_model.model_["selection"])
print("selected orders:", full_model.selected_orders_)
print("elapsed seconds:", round(full_elapsed, 2))
print("48-step RMSE:", round(full_rmse, 4))
pd.DataFrame(full_fcst).head()


## Cheaper Restricted Search

A practical first pass is to remove combinations that are unlikely to pay off. The version below still models daily and weekly seasonality, but it fixes non-seasonal AR and differencing to zero, keeps MA terms off, and keeps constants off. This reduces the search from 31 candidate fits to 7.

Use this kind of restricted search when the broad auto model is too slow, then widen one dimension at a time only if diagnostics or accuracy justify it.


In [ ]:
cheap_model = AutoMSARIMAX(
    lags=[1, 24, 168],
    max_ar_order=[0, 1, 1],
    max_i_order=[0, 1, 0],
    max_ma_order=[0, 0, 0],
    include_constant=False,
    n_jobs=-1,
)

start = time.perf_counter()
cheap_model.fit(train_y)
cheap_fcst = cheap_model.predict(h=h, level=[80, 95])
cheap_elapsed = time.perf_counter() - start
cheap_rmse = np.sqrt(np.mean((cheap_fcst["mean"] - test) ** 2))

print("selection:", cheap_model.model_["selection"])
print("selected orders:", cheap_model.selected_orders_)
print("elapsed seconds:", round(cheap_elapsed, 2))
print("48-step RMSE:", round(cheap_rmse, 4))
pd.DataFrame(cheap_fcst).head()
